In [1]:
!pip install azure-ai-projects azure-core azure-storage-blob

In [2]:
pip install --upgrade openai azure-ai-projects azure-identity

   ---------------------------------------- 0.0/1.2 MB ? eta -:--:--
   ---------------------------------------- 1.2/1.2 MB 14.5 MB/s  0:00:00

  Attempting uninstall: openai

    Found existing installation: openai 2.29.0

   ---------------------------------------- 0/2 [openai]
   ---------------------------------------- 0/2 [openai]
   ---------------------------------------- 0/2 [openai]
    Uninstalling openai-2.29.0:
   ---------------------------------------- 0/2 [openai]
      Successfully uninstalled openai-2.29.0
   ---------------------------------------- 0/2 [openai]
   ---------------------------------------- 0/2 [openai]
   ---------------------------------------- 0/2 [openai]
   ---------------------------------------- 0/2 [openai]
   ---------------------------------------- 0/2 [openai]
   ---------------------------------------- 0/2 [openai]
   ---------------------------------------- 0/2 [openai]
   ---------------------------------------- 0/2 [openai]
   ------------

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
crewai 1.11.0 requires chromadb~=1.1.0, but you have chromadb 1.5.5 which is incompatible.


In [ ]:
from openai import OpenAI
import os
from dotenv import load_dotenv

load_dotenv()
client = OpenAI(
    api_key=os.getenv('FOUNDARY_API_KEY'),
    # With the standard OpenAI client, the /openai/v1 suffix is REQUIRED
    base_url="https://ai-foundary-demo2-resource.openai.azure.com/openai/v1"
)

# Note: api_version is usually NOT required when using this path in 2026

In [22]:
# 1. Define the input data
# In a real scenario, you can load this from a .txt or .json file
resume_data = """
RICHARD SANCHEZ
Education: Master of Business Management (2029-2030)
Work: Marketing Manager at Borcelle Studio (2030-Present)
Skills: Project Management, PR, Leadership...
"""

# 2. Use the Responses API for structured extraction
# Use the structured input pattern for GPT-4.1
# Updated call with 2026-compliant type naming
try:
    response = client.responses.create(
        model="gpt-4.1",
        input=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "input_text",  # Changed from 'text'
                        "text": "Extract the following details into a JSON object: Name, Education, Latest Job, and Top 3 Skills. Return ONLY valid JSON."
                    },
                    {
                        "type": "input_text",  # Changed from 'text'
                        "text": f"RESUME SOURCE:\n{resume_data}"
                    }
                ]
            }
        ]
    )

    print("--- EXTRACTED DATA ---")
    print(response.output[0].content)

except Exception as e:
    print(f"Extraction Error: {e}")

--- EXTRACTED DATA ---
[ResponseOutputText(annotations=[], text='{\n  "Name": "Richard Sanchez",\n  "Education": "Master of Business Management (2029-2030)",\n  "Latest Job": "Marketing Manager at Borcelle Studio (2030-Present)",\n  "Top 3 Skills": ["Project Management", "PR", "Leadership"]\n}', type='output_text', logprobs=[])]


In [15]:
import json

try:
    # 1. Get the first item in the content list
    # 2. Access the 'text' attribute of that item
    output_text_item = response.output[0].content[0]

    # In some SDK versions, it's a dict; in others, it's an object.
    # This handles both:
    if isinstance(output_text_item, dict):
        raw_json_string = output_text_item.get("text", "")
    else:
        raw_json_string = output_text_item.text

    # Now parse the actual string
    extracted_data = json.loads(raw_json_string)

    print("--- SUCCESS: DATA PARSED ---")
    print(json.dumps(extracted_data, indent=2))

except Exception as e:
    print(f"Parsing Error: {e}")
    # Troubleshooting: print the type to see what exactly was returned
    print(f"Actual content type: {type(response.output[0].content[0])}")

Parsing Error: Expecting value: line 1 column 1 (char 0)
Actual content type: <class 'openai.types.responses.response_output_text.ResponseOutputText'>


In [ ]:
from azure.storage.blob import BlobServiceClient
from dotenv import load_dotenv
import os

load_dotenv()
# ✅ Correct connection string
AZURE_STORAGE_CONNECTION_STRING = os.getenv('AZURE_CONNECTION_STRING')

blob_service_client = BlobServiceClient.from_connection_string(
    AZURE_STORAGE_CONNECTION_STRING
)

container_name = "akcontainers"

blob_client = blob_service_client.get_blob_client(
    container=container_name,
    blob="result.txt"
)

blob_client.upload_blob("Test upload", overwrite=True)

print("✅ Upload successful")

✅ Upload successful


In [25]:
import datetime

file_name = f"resume_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"

In [26]:
blob_client = blob_service_client.get_blob_client(
    container="akcontainers",
    blob=file_name
)

blob_client.upload_blob(response.output_text, overwrite=True)

{'etag': '"0x8DE9F7048B33C71"',
 'last_modified': datetime.datetime(2026, 4, 21, 6, 36, 13, tzinfo=datetime.timezone.utc),
 'content_md5': bytearray(b'\xdb\xe3g\xe1$0\xd2G\xd7k\xff\x97\xbd\xe3\xdb`'),
 'client_request_id': '63d15c1e-3d4c-11f1-aee6-e8d8d1a8e59c',
 'request_id': '29b4ca3f-901e-0012-2259-d1f0b7000000',
 'version': '2026-02-06',
 'version_id': None,
 'date': datetime.datetime(2026, 4, 21, 6, 36, 13, tzinfo=datetime.timezone.utc),
 'request_server_encrypted': True,
 'encryption_key_sha256': None,
 'encryption_scope': None,
 'structured_body': None}

In [27]:
print(f"✅ Stored as: {file_name}")

✅ Stored as: resume_20260421_120610.txt
